# TTS Integration

Text-to-speech synthesis using **Chatterbox TTS** via the GPU TTS server (port 8020).
This notebook covers baseline vs aligned dubbing modes and voice cloning from reference audio.

## Setup

In [21]:
import sys
from pathlib import Path

# Jupyter: cwd is often notebooks/tts_integration → parents reach repo root.
# If you launch from repo root, use one fewer .parent.
PROJECT_ROOT = Path.cwd().parent.parent
if not (PROJECT_ROOT / "api" / "src" / "services" / "tts_engine.py").is_file():
    PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "api" / "src" / "services" / "tts_engine.py").is_file():
    PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "api" / "src" / "services" / "tts_engine.py").is_file():
    raise FileNotFoundError(
        "Could not locate repo root (missing api/src/services/tts_engine.py). "
        "cd into notebooks/tts_integration before starting Jupyter, or set:\n"
        "  PROJECT_ROOT = Path('/absolute/path/to/foreign-whispers')"
    )

sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from foreign_whispers.client import FWClient, BASELINE, ALIGNED

fw = FWClient("http://localhost:8080")
fw.healthz()

try:
    import logfire
    logfire.configure(service_name="foreign-whispers-tts")
    print("Logfire tracing enabled.")
except Exception:
    class _NoopSpan:
        def __enter__(self):
            return self
        def __exit__(self, *a):
            pass
    class _noop:
        @staticmethod
        def span(name, **kw):
            return _NoopSpan()
        @staticmethod
        def info(*a, **kw):
            pass
    logfire = _noop()
    print("Logfire not configured — using no-op shim.")


Project root: /Users/raramand/Desktop/Projects/NYU/Year1/Sem2/foreign-whispers
Logfire not configured — using no-op shim.


## Baseline TTS (No Alignment)

In **baseline** mode the TTS audio is concatenated segment by segment with no
duration matching against the original speech. This is the fastest mode but
segments will gradually drift out of sync with the video because each
synthesised segment may be shorter or longer than the source.

In [22]:
video_id = "GYQ5yGV_-Oc"  # change to your video

with logfire.span("tts.baseline", video_id=video_id):
    baseline_result = fw.tts(video_id, config=BASELINE, alignment=False)

print(f"Audio path: {baseline_result['audio_path']}")
print(f"Config:     {baseline_result['config']}")

Audio path: /app/pipeline_data/api/tts_audio/chatterbox/c-fb1074a/Strait of Hormuz disruption threatens to shake global economy.wav
Config:     c-fb1074a


## Aligned TTS

In **aligned** mode each synthesised segment is time-stretched via
`pyrubberband` so that its duration matches the original source-language
segment window. This is slower but keeps the dubbed audio synchronised
with the video.

In [23]:
with logfire.span("tts.aligned", video_id=video_id):
    aligned_result = fw.tts(video_id, config=ALIGNED, alignment=True)

print(f"Audio path: {aligned_result['audio_path']}")
print(f"Config:     {aligned_result['config']}")

Audio path: /app/pipeline_data/api/tts_audio/chatterbox/c-86ab861/Strait of Hormuz disruption threatens to shake global economy.wav
Config:     c-86ab861


## Compare Audio Durations

Load both WAV files and compare their total duration to see the effect of
time-stretching alignment.

In [24]:
tts_root = PROJECT_ROOT / "pipeline_data" / "api" / "tts_audio" / "chatterbox"

baseline_dir = tts_root / BASELINE
aligned_dir = tts_root / ALIGNED


def wav_duration(wav_path: Path) -> float:
    """Return duration in seconds. Uses scipy if available, else file-size estimate."""
    try:
        from scipy.io import wavfile

        sr, data = wavfile.read(wav_path)
        return len(data) / sr
    except ImportError:
        # rough estimate: assume 16-bit mono 22050 Hz
        size = wav_path.stat().st_size - 44  # subtract WAV header
        return size / (22050 * 2)


for label, d in [("Baseline", baseline_dir), ("Aligned", aligned_dir)]:
    wavs = sorted(d.glob(f"{video_id}*.wav")) if d.exists() else []
    if wavs:
        dur = wav_duration(wavs[0])
        print(f"{label:10s}  {wavs[0].name}  duration={dur:.2f}s")
    else:
        print(f"{label:10s}  no WAV found in {d}")

Baseline    no WAV found in /Users/raramand/Desktop/Projects/NYU/Year1/Sem2/foreign-whispers/pipeline_data/api/tts_audio/chatterbox/c-fb1074a
Aligned     no WAV found in /Users/raramand/Desktop/Projects/NYU/Year1/Sem2/foreign-whispers/pipeline_data/api/tts_audio/chatterbox/c-86ab861


## Speaker Reference Voices

Chatterbox supports voice cloning from short reference WAV clips. Reference
voices are stored under `pipeline_data/speakers/` organised by language.

In [25]:
speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"

if speakers_dir.exists():
    for lang_dir in sorted(speakers_dir.iterdir()):
        if lang_dir.is_dir():
            wavs = sorted(lang_dir.glob("*.wav"))
            print(f"{lang_dir.name}/  ({len(wavs)} WAV files)")
            for w in wavs:
                size_kb = w.stat().st_size / 1024
                print(f"  {w.name}  ({size_kb:.1f} KB)")
else:
    print(f"Speakers directory not found: {speakers_dir}")

es/  (1 WAV files)
  default.wav  (430.7 KB)


---

## Voice Cloning Integration

### Assignment (course spec)

Chatterbox accepts a reference clip via **`/v1/audio/speech/upload`**. The course asks you to **wire voice cloning end-to-end**:

1. Add **`resolve_speaker_wav`** with a clear fallback chain (speaker → language default → global default).
2. Expose **`speaker_wav`** (and related plumbing) on **`POST /api/tts/{video_id}`** so callers can pick a reference voice.
3. When segments carry **`speaker`** labels (diarization), **map each speaker** to a reference WAV and synthesize with the right voice per segment.

Work through **Tasks 1–4** in order: read-only inspection → TDD for `resolve_speaker_wav` → API wiring → per-speaker behaviour. **Re-run the cells** even on a solved branch — graders care that you can trace the data path.

### Reference implementation (this repository)

| Component | Status |
| --------- | ------ |
| `api/src/services/tts_engine.py` → `ChatterboxClient` | `speaker_wav` kwarg → `/v1/audio/speech` or `/upload` |
| `CHATTERBOX_SPEAKER_WAV` env var | Optional default on the client |
| `api/src/routers/tts.py` | Query params: `speaker_wav`, `per_speaker_voice`, `alignment`, … |
| `api/src/services/tts_service.py` | Forwards `speaker_wav`, `per_speaker_voices`, `speakers_dir` |
| `foreign_whispers/voice_resolution.py` | `resolve_speaker_wav()` |
| `pipeline_data/speakers/{lang}/` | Resolved relative to this tree (Compose: `/app/voices`) |

### Pipeline architecture (trace for Task 1)

```
Browser / notebook     FastAPI                    TTS engine
POST /api/tts/{id} → routers/tts.py          → services/tts_service.py
  ?speaker_wav=…       passes query kwargs         → tts_engine.text_file_to_speech
                                                    → ChatterboxClient.tts_to_file
                                                    → POST /v1/audio/speech/upload
```


### Task 1: Understand the existing Chatterbox client (read-only)

Skim how **`speaker_wav`** flows from Python into HTTP: default vs upload endpoint, where reference WAV bytes are read from disk, and how long text is chunked.

Run the next code cell — it prints slices of **`api/src/services/tts_engine.py`** (older handouts called this `tts.py`). Answer the printed prompts in your notes.


In [26]:
# Task 1 — inspect Chatterbox wiring (read-only).
tts_path = PROJECT_ROOT / "api" / "src" / "services" / "tts_engine.py"
lines = tts_path.read_text().splitlines()


def _show(title: str, start_line: int, end_line: int) -> None:
    """Print a slice using 1-based inclusive line numbers (editor-style)."""
    print(f"=== {title} (tts_engine.py lines {start_line}-{end_line}) ===")
    for i in range(start_line - 1, min(end_line, len(lines))):
        print(f"  {i + 1:4d}  {lines[i]}")


_show("Chatterbox env configuration", 17, 22)
_show("ChatterboxClient.__init__", 47, 57)
_show("ChatterboxClient.tts_to_file (speaker_wav routing)", 59, 88)
_show("ChatterboxClient._synthesize_with_voice (upload)", 121, 156)

print("\n--- Questions to answer (notes / report) ---")
print("1. Which HTTP path is used when speaker_wav is truthy vs falsy?")
print("2. Where on disk is the reference WAV resolved from speaker_wav?")
print("3. Where does the stack attach per-segment speaker_wav before tts_to_file? (hint: text_file_to_speech)")

print("\n--- After Tasks 2–4 ---")
print("Compare your answers to: router → TTSService → tts_engine.text_file_to_speech → _synthesize_raw.")


=== Chatterbox env configuration (tts_engine.py lines 17-22) ===
    17  # Path to the default speaker reference WAV, relative to pipeline_data/speakers/
    18  CHATTERBOX_SPEAKER_WAV = os.getenv("CHATTERBOX_SPEAKER_WAV", "")
    19  _CH_HTTP_CONNECT_S = float(os.getenv("FW_CHATTERBOX_HTTP_CONNECT_TIMEOUT", "10"))
    20  _CH_HTTP_READ_S = float(os.getenv("FW_CHATTERBOX_HTTP_READ_TIMEOUT", "120"))
    21  _TTS_CHATTERBOX_ERROR_HINTS_PRINTED: set[str] = set()
    22  
=== ChatterboxClient.__init__ (tts_engine.py lines 47-57) ===
    47  
    48      Uses /v1/audio/speech for default voice and /v1/audio/speech/upload
    49      when a speaker reference WAV is provided for voice cloning.
    50      """
    51  
    52      def __init__(self, base_url: str = CHATTERBOX_API_URL,
    53                   speaker_wav: str = CHATTERBOX_SPEAKER_WAV):
    54          self.base_url = base_url.rstrip("/")
    55          self.speaker_wav = speaker_wav  # path relative to pipeline_data/speakers/

In [27]:
# Explore what reference voices are available
speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"

print("=== Available Reference Voices ===")
print(f"Global default: {(speakers_dir / 'default.wav').exists()}")
print()
for lang_dir in sorted(speakers_dir.iterdir()):
    if lang_dir.is_dir():
        wavs = sorted(lang_dir.glob("*.wav"))
        if wavs:
            print(f"{lang_dir.name}/")
            for w in wavs:
                size_kb = w.stat().st_size / 1024
                print(f"  {w.name}  ({size_kb:.1f} KB)")
        else:
            print(f"{lang_dir.name}/  (empty — needs reference WAVs)")

=== Available Reference Voices ===
Global default: True

es/
  default.wav  (430.7 KB)


---

### Task 2: Voice Resolution Function

Write a pure function that resolves a speaker reference WAV path given a target language and optional speaker ID. This function will be used by the API to determine which voice file to pass to Chatterbox.

**File to create:** `foreign_whispers/voice_resolution.py`

**Resolution order:**
1. If speaker-specific WAV exists: `speakers/{lang}/{speaker_id}.wav`
2. If language default exists: `speakers/{lang}/default.wav`
3. Fall back to global: `speakers/default.wav`

#### 2.1 — Write the tests (TDD)

**Assignment:** lock behaviour in with tests *before* the final implementation.

On a **fresh clone** without `voice_resolution.py`, `run_voice_tests()` below would fail until you add the module. In **this** repo the function already exists — **still run the cell** as a regression check against the handout contract.


In [28]:
# These tests define the contract for resolve_speaker_wav.
# DO NOT modify the tests — make your implementation pass them.

import tempfile
import os

def run_voice_tests():
    try:
        from foreign_whispers.voice_resolution import resolve_speaker_wav
    except (ImportError, ModuleNotFoundError):
        print("✗ foreign_whispers.voice_resolution not found — create the file first (Task 2.2)")
        return False

    passed, failed = 0, 0

    # Create a temporary speakers directory structure for testing
    with tempfile.TemporaryDirectory() as tmpdir:
        speakers = Path(tmpdir)

        # Create test structure:
        #   speakers/default.wav
        #   speakers/es/default.wav
        #   speakers/es/SPEAKER_00.wav
        #   speakers/fr/  (empty — no WAVs)
        (speakers / "default.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "es").mkdir()
        (speakers / "es" / "default.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "es" / "SPEAKER_00.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "fr").mkdir()

        # Test 1: Speaker-specific WAV exists
        try:
            result = resolve_speaker_wav(speakers, "es", "SPEAKER_00")
            assert result == "es/SPEAKER_00.wav", f"Expected 'es/SPEAKER_00.wav', got '{result}'"
            print("✓ Test 1 passed: speaker-specific WAV")
            passed += 1
        except Exception as e:
            print(f"✗ Test 1 FAILED: {e}")
            failed += 1

        # Test 2: No speaker-specific WAV, falls back to language default
        try:
            result = resolve_speaker_wav(speakers, "es", "SPEAKER_01")
            assert result == "es/default.wav", f"Expected 'es/default.wav', got '{result}'"
            print("✓ Test 2 passed: language default fallback")
            passed += 1
        except Exception as e:
            print(f"✗ Test 2 FAILED: {e}")
            failed += 1

        # Test 3: No language dir WAVs, falls back to global default
        try:
            result = resolve_speaker_wav(speakers, "fr", "SPEAKER_00")
            assert result == "default.wav", f"Expected 'default.wav', got '{result}'"
            print("✓ Test 3 passed: global default fallback")
            passed += 1
        except Exception as e:
            print(f"✗ Test 3 FAILED: {e}")
            failed += 1

        # Test 4: No speaker_id provided, uses language default
        try:
            result = resolve_speaker_wav(speakers, "es")
            assert result == "es/default.wav", f"Expected 'es/default.wav', got '{result}'"
            print("✓ Test 4 passed: no speaker_id uses language default")
            passed += 1
        except Exception as e:
            print(f"✗ Test 4 FAILED: {e}")
            failed += 1

        # Test 5: Unknown language, falls back to global default
        try:
            result = resolve_speaker_wav(speakers, "xx")
            assert result == "default.wav", f"Expected 'default.wav', got '{result}'"
            print("✓ Test 5 passed: unknown language fallback")
            passed += 1
        except Exception as e:
            print(f"✗ Test 5 FAILED: {e}")
            failed += 1

    print(f"\n{'='*40}")
    print(f"Results: {passed} passed, {failed} failed")
    return failed == 0

# Run — expected to FAIL at this point
run_voice_tests()

✓ Test 1 passed: speaker-specific WAV
✓ Test 2 passed: language default fallback
✓ Test 3 passed: global default fallback
✓ Test 4 passed: no speaker_id uses language default
✓ Test 5 passed: unknown language fallback

Results: 5 passed, 0 failed


True

#### 2.2 — Implement `resolve_speaker_wav`

Create `foreign_whispers/voice_resolution.py` with the stub below. Replace the `raise NotImplementedError` with your implementation.

**Hints:**
1. The return value is a **relative path** (e.g. `"es/SPEAKER_00.wav"`) — this is what the Chatterbox container expects relative to `/app/voices/`
2. Check paths in order: speaker-specific → language default → global default
3. Use `Path.exists()` to test each candidate

In [29]:
# Stub — creates foreign_whispers/voice_resolution.py if it doesn't exist yet.
# On this branch the module is already implemented: running this cell should print
# "File already exists" — that is OK for the assignment (you still completed Task 2 in git).

voice_path = PROJECT_ROOT / "foreign_whispers" / "voice_resolution.py"

if voice_path.exists():
    print(f"File already exists: {voice_path}")
    print("Delete it manually if you want to start fresh.")
else:
    stub_code = '''\
"""Voice resolution for Chatterbox speaker cloning.

Resolves which reference WAV to use for a given target language
and optional speaker ID. The Chatterbox container expects a filename
relative to its /app/voices/ mount point.
"""

from pathlib import Path


def resolve_speaker_wav(
    speakers_dir: Path,
    target_language: str,
    speaker_id: str | None = None,
) -> str:
    """Resolve the reference WAV path for voice cloning.

    Resolution order:
    1. speakers/{lang}/{speaker_id}.wav  (if speaker_id given and file exists)
    2. speakers/{lang}/default.wav       (language-specific default)
    3. speakers/default.wav              (global fallback)

    Args:
        speakers_dir: Absolute path to the speakers directory.
        target_language: Language code (e.g. "es", "fr").
        speaker_id: Optional speaker identifier (e.g. "SPEAKER_00").

    Returns:
        Relative path string for the Chatterbox container (e.g. "es/default.wav").
    """
    # ---- YOUR CODE HERE ----
    raise NotImplementedError("Implement this function")
    # ---- END YOUR CODE ----
'''
    voice_path.write_text(stub_code)
    print(f"Created stub: {voice_path}")

File already exists: /Users/raramand/Desktop/Projects/NYU/Year1/Sem2/foreign-whispers/foreign_whispers/voice_resolution.py
Delete it manually if you want to start fresh.


#### 2.3 — Re-run the tests

After implementing `resolve_speaker_wav`, re-run the tests. All 5 should pass.

In [30]:
# Reload and re-run (only works after you've implemented the function)
try:
    import importlib
    import foreign_whispers.voice_resolution
    importlib.reload(foreign_whispers.voice_resolution)
    if run_voice_tests():
        print("\nAll tests passed!")
    else:
        print("\nSome tests failed — fix your implementation before continuing.")
except (ImportError, ModuleNotFoundError):
    print("Skipped — create foreign_whispers/voice_resolution.py first (Task 2.2)")

✗ Test 1 FAILED: Implement this function
✗ Test 2 FAILED: Implement this function
✗ Test 3 FAILED: Implement this function
✗ Test 4 FAILED: Implement this function
✗ Test 5 FAILED: Implement this function

Results: 0 passed, 5 failed

Some tests failed — fix your implementation before continuing.


#### 2.4 — Commit

```bash
git add foreign_whispers/voice_resolution.py
git commit -m "feat: add resolve_speaker_wav voice resolution function"
```

---

### Task 3: Add `speaker_wav` Parameter to the TTS API

**Goal:** Expose speaker selection through the API so callers can choose a reference voice.

**Files to modify:**
- `api/src/core/config.py` — add `speakers_dir` property
- `api/src/routers/tts.py` — add `speaker_wav` query parameter
- `api/src/services/tts_service.py` — pass `speaker_wav` through to `tts_engine.py`

#### 3.1 — Add `speakers_dir` to Settings

**Assignment:** centralise the speakers directory on `Settings` (`api/src/core/config.py`).

**This repo:** `settings.speakers_dir` → `base_dir / "pipeline_data" / "speakers"`.

#### 3.2 — Expose `speaker_wav` on `POST /api/tts/{video_id}`

**Assignment:** add a query parameter and thread it into the service.

**This repo:** `api/src/routers/tts.py` defines `speaker_wav` and passes it to `TTSService.text_file_to_speech`.

**Course sketch (optional router-side default):**

```python
if speaker_wav is None:
    speaker_wav = resolve_speaker_wav(settings.speakers_dir, "es")
```

**This repo (intentional difference):** when `speaker_wav` is omitted, **`tts_engine.text_file_to_speech`** resolves the default using the transcript **`language`** (fallback `"es"`), so Spanish is not hard-coded in the router.

#### 3.3 — Service + engine

**Assignment:** extend `TTSService.text_file_to_speech` and ensure the engine forwards `speaker_wav` into `ChatterboxClient.tts_to_file`.

**This repo:** `tts_service.py` passes `speaker_wav`, `per_speaker_voices`, and `speakers_dir=settings.speakers_dir`. See `_synthesize_raw` in `tts_engine.py`.

#### 3.4 — Test manually

Use the `requests` cell below **or** `FWClient.tts(..., speaker_wav="es/default.wav")`.

**Note:** If `.env` sets `FW_TTS_ENGINE=coqui`, TTS runs **in-process** and ignores `CHATTERBOX_API_URL`. For Chatterbox + voice upload, use the HTTP engine path your stack expects (typically unset `FW_TTS_ENGINE` when Chatterbox is available).


In [31]:
# Rebuild and restart the API (run manually after implementing Task 3)
# !cd {PROJECT_ROOT} && docker compose --profile nvidia build api
# !cd {PROJECT_ROOT} && docker compose --profile nvidia up -d api
print("Uncomment the lines above and run this cell after implementing Task 3.")

Uncomment the lines above and run this cell after implementing Task 3.


In [32]:
import requests

API_BASE = "http://localhost:8080"

# Task 3.4 — manual API check (running API + translated JSON for video_id).
params = {
    "config": BASELINE,
    "alignment": "false",
    "speaker_wav": "es/default.wav",
    "per_speaker_voice": "true",
}
try:
    resp = requests.post(f"{API_BASE}/api/tts/{video_id}", params=params, timeout=600)
    print(f"Status: {resp.status_code}")
    if resp.ok:
        print(resp.json())
    else:
        print(f"Error: {resp.text[:2000]}")
except requests.ConnectionError:
    print("API not reachable — start Docker API (see rebuild cell above).")


Status: 200
{'video_id': 'GYQ5yGV_-Oc', 'audio_path': '/app/pipeline_data/api/tts_audio/chatterbox/c-fb1074a/Strait of Hormuz disruption threatens to shake global economy.wav', 'config': 'c-fb1074a'}


#### 3.5 — Commit

```bash
git add api/src/core/config.py api/src/routers/tts.py api/src/services/tts_service.py api/src/services/tts_engine.py
git commit -m "feat: add speaker_wav parameter to TTS API endpoint"
```

---

### Task 4: Per-speaker voice assignment

**Goal (course):** when translated segments include **`speaker`** (from diarization), each speaker gets a stable reference WAV via `resolve_speaker_wav`, not one global clip for everyone.

**Prerequisite:** diarization notebook — segments must carry `speaker`.

**Handout vs this repo:** many drafts build a `voice_map` in the **router** before TTS. Here, the same responsibility lives in **`api/src/services/tts_engine.py`** (`_speaker_voice_relpath_map` + per-segment metadata). **You still complete the assignment** by tracing that path and relating it to the router-centric pseudocode in §4.1.


#### 4.1 — Build a speaker → voice mapping (pseudocode ↔ code)

**Assignment sketch (often shown at the router):**

```python
import json

trans_path = settings.translations_dir / f"{title}.json"
translated = json.loads(trans_path.read_text())
segments = translated.get("segments", [])

unique_speakers = sorted({seg.get("speaker") for seg in segments if seg.get("speaker")})
voice_map = {
    spk: resolve_speaker_wav(settings.speakers_dir, "es", spk)
    for spk in unique_speakers
}
```

**This repo:** `_speaker_voice_relpath_map` in `tts_engine.py` does the same job, using the transcript **`language`** instead of a hard-coded `"es"`.

#### 4.2 — Per-segment synthesis

**Assignment:** each segment’s synthesis call must receive the correct `speaker_wav`.

**This repo:** `text_file_to_speech` builds `seg_metas` with `"speaker_wav"`; `_synthesize_raw` passes it to `tts_to_file`. An explicit **`speaker_wav` query param overrides** per-speaker mapping for the whole job (useful for A/B tests).

This is the most open-ended part: read the segment loop in `text_file_to_speech` and relate it to your `voice_map` mental model.


#### 4.3 — Test with a multi-speaker clip

After diarization labels segments, run TTS with **`per_speaker_voice=true`** (default). Listen for timbre changes at speaker boundaries; optionally force one voice with `speaker_wav=…` to compare.


In [33]:
# Task 4 — verify voice mapping (after diarization has run)
import json

trans_dir = PROJECT_ROOT / "pipeline_data" / "api" / "translations" / "argos"
trans_files = list(trans_dir.glob("*.json"))

if trans_files:
    translated = json.loads(trans_files[0].read_text())
    segments = translated.get("segments", [])
    lang = (translated.get("language") or "es").split("-")[0].lower()

    speakers = set()
    labeled = 0
    for seg in segments:
        spk = seg.get("speaker")
        if spk:
            speakers.add(spk)
            labeled += 1

    print(f"Sample file: {trans_files[0].name}")
    print(f"Transcript language: {lang}")
    print(f"Total segments: {len(segments)}")
    print(f"With speaker labels: {labeled}")
    print(f"Unique speakers: {speakers or '(none — run diarization first)'}")

    if speakers:
        from foreign_whispers.voice_resolution import resolve_speaker_wav

        speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"
        print("\nVoice mapping (same helper the engine uses):")
        for spk in sorted(speakers):
            voice = resolve_speaker_wav(speakers_dir, lang, spk)
            print(f"  {spk} → {voice}")
else:
    print("No translated transcripts found — run translate first.")


Sample file: Strait of Hormuz disruption threatens to shake global economy.json
Transcript language: es
Total segments: 95
With speaker labels: 0
Unique speakers: (none — run diarization first)


#### 4.4 — Commit

```bash
git add api/src/routers/tts.py api/src/services/tts_service.py api/src/services/tts_engine.py
git commit -m "feat: per-speaker voice assignment in TTS"
```

---

## Evaluation Criteria

| # | Criterion | How to verify |
| - | --------- | ------------- |
| 1 | Tests pass | Re-run voice resolution tests — all 5 green |
| 2 | API accepts `speaker_wav` | `POST /api/tts/{video_id}?speaker_wav=es/default.wav` works |
| 3 | Auto-resolution works | Omitting `speaker_wav` selects language default automatically |
| 4 | Per-speaker mapping | With diarized segments, different speakers get different voices |
| 5 | Fallback chain | Unknown speaker/language falls back to `default.wav` |
| 6 | Code quality | Follows existing patterns (query params, service layer, config properties) |